In [1]:
!pip install gradio transformers torch -q
print("Done")

Done


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
import json

save_dir = "/content/drive/MyDrive/VaakSetu"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load label mappings
checkpoint = torch.load(f"{save_dir}/vaaksetu_final_model.pt",
                        map_location=device, weights_only=False)

INTENT2ID = checkpoint["intent2id"]
ID2INTENT = checkpoint["id2intent"]
NER2ID = checkpoint["ner2id"]
ID2NER = checkpoint["id2ner"]

print("Mappings loaded")
print(f"Intents: {list(INTENT2ID.keys())}")

Mounted at /content/drive
Mappings loaded
Intents: ['SEARCH', 'COMPARE', 'BUY', 'TRACK', 'RETURN']


In [4]:
class VaakSetuModel(nn.Module):
    def __init__(self, num_intents=5, num_ner_labels=13):
        super().__init__()
        self.muril = AutoModel.from_pretrained("google/muril-base-cased")
        hidden_size = self.muril.config.hidden_size
        self.intent_head = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(hidden_size, num_intents)
        )
        self.ner_head = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(hidden_size, num_ner_labels)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.muril(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        cls_output = outputs.last_hidden_state[:, 0, :]
        sequence_output = outputs.last_hidden_state
        intent_logits = self.intent_head(cls_output)
        ner_logits = self.ner_head(sequence_output)
        return intent_logits, ner_logits

tokenizer = AutoTokenizer.from_pretrained("google/muril-base-cased")
model = VaakSetuModel().to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Model loaded and ready")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded and ready


In [5]:
import gradio as gr

def predict(query):
    if not query.strip():
        return "Please enter a query", ""

    words = query.strip().split()

    encoding = tokenizer(
        words,
        is_split_into_words=True,
        max_length=128,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        intent_logits, ner_logits = model(
            encoding["input_ids"],
            encoding["attention_mask"]
        )

    # Intent prediction
    intent_id = torch.argmax(intent_logits, dim=1).item()
    intent = ID2INTENT[intent_id]

    # NER prediction
    word_ids = encoding.word_ids(batch_index=0)
    ner_predictions = torch.argmax(ner_logits, dim=2).squeeze(0)

    entities = []
    current_entity = None
    current_type = None

    for idx, word_id in enumerate(word_ids):
        if word_id is None:
            continue
        if word_id >= len(words):
            continue

        label = ID2NER[ner_predictions[idx].item()]
        word = words[word_id]

        if label.startswith("B-"):
            if current_entity:
                entities.append((current_entity, current_type))
            current_entity = word
            current_type = label[2:]
        elif label.startswith("I-") and current_entity:
            current_entity += " " + word
        else:
            if current_entity:
                entities.append((current_entity, current_type))
                current_entity = None
                current_type = None

    if current_entity:
        entities.append((current_entity, current_type))

    # Remove duplicates
    seen = set()
    unique_entities = []
    for entity, etype in entities:
        if entity not in seen:
            seen.add(entity)
            unique_entities.append((entity, etype))

    # Format output
    intent_output = f"🎯 Intent: {intent}"

    if unique_entities:
        entity_lines = "\n".join([f"  • {text} → {etype}"
                                   for text, etype in unique_entities])
        entity_output = f"📦 Entities Found:\n{entity_lines}"
    else:
        entity_output = "📦 Entities Found: None"

    return intent_output, entity_output

# Build Gradio interface
demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(
        label="Enter Hinglish Query",
        placeholder="e.g. Nike wala red kurti size M teen hazaar ke andar chahiye",
        lines=2
    ),
    outputs=[
        gr.Textbox(label="Intent"),
        gr.Textbox(label="Entities")
    ],
    title="VaakSetu — Hinglish NLU for Indian E-commerce",
    description="Type a Hinglish product search query. VaakSetu will identify the intent and extract entities.",
    examples=[
        ["Nike wala red kurti size M teen hazaar ke andar chahiye fast delivery"],
        ["Samsung aur OnePlus mein camera better kiska hai"],
        ["mera order kahan hai 3 din ho gaye"],
        ["iPhone 16 abhi buy karna hai"],
        ["yeh shoes return karna hai size fit nahi hua"]
    ]
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6d4181bb199386ad53.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
